<a href="https://colab.research.google.com/github/dtabuena/Resources/blob/main/Genetics/Gene_translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
Gene translator sandbox - clean notebook for building and testing
"""
import os
import json
import time
import warnings
import urllib.request
import urllib.error
import pandas as pd
import numpy as np

In [6]:
"""
GeneTranslator: MGI-backed mouse gene symbol reconciliation across reference vintages.

Build once, then use to translate symbols between datasets aligned to different
Ensembl/GENCODE releases. Default backbone is MGI MRK_List1.rpt which contains
both official and withdrawn symbols with their forward replacement pointers.

Bridge logic:
  - any symbol (modern or legacy) -> uppercase lookup -> MGI ID
  - MGI ID -> current_symbol (if locus is still in use)
  - withdrawn symbols carry a forward pointer to their replacement MGI ID
  - aliases (Marker Synonyms) catch renames not handled by the withdraw list

Persisted artifacts in cache_dir:
  - mgi_master.parquet      one row per MGI ID, current symbol, all synonyms
  - symbol_lookup.parquet   one row per (UPPERCASE_SYMBOL, mgi_id) edge
  - meta.json               build timestamp, source URL, file mtime
"""
import os
import json
import time
import warnings
import urllib.request
import urllib.error
import pandas as pd
import numpy as np


class GeneTranslator:

    MGI_BASE_URL = 'https://www.informatics.jax.org/downloads/reports'
    MRK_LIST1_FILE = 'MRK_List1.rpt'

    MRK_LIST1_COLUMNS = [
        'mgi_id',
        'chromosome',
        'cm_position',
        'genome_start',
        'genome_end',
        'strand',
        'symbol',
        'status',
        'name',
        'marker_type',
        'feature_types',
        'synonyms',
        'mgi_id_current',
        'symbol_current',
    ]

    def __init__(self, cache_dir):
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)
        self.master_path  = os.path.join(self.cache_dir, 'mgi_master.parquet')
        self.lookup_path  = os.path.join(self.cache_dir, 'symbol_lookup.parquet')
        self.meta_path    = os.path.join(self.cache_dir, 'meta.json')
        self.raw_path     = os.path.join(self.cache_dir, self.MRK_LIST1_FILE)

        self.master = None  # DataFrame indexed by mgi_id
        self.lookup = None  # DataFrame indexed by symbol_upper

    # ---------- BUILD ----------

    def is_built(self):
        return (os.path.exists(self.master_path) and
                os.path.exists(self.lookup_path) and
                os.path.exists(self.meta_path))

    def build(self, force=False):
        if self.is_built() and not force:
            print(f'[build] cache exists at {self.cache_dir}, skipping (force=False)')
            return self.load()

        print(f'[build] downloading {self.MRK_LIST1_FILE}...')
        url = f'{self.MGI_BASE_URL}/{self.MRK_LIST1_FILE}'
        try:
            urllib.request.urlretrieve(url, self.raw_path)
        except urllib.error.URLError as exc:
            raise RuntimeError(f'MGI download failed from {url}: {exc!r}')

        size_mb = os.path.getsize(self.raw_path) / 1e6
        print(f'[build] downloaded {size_mb:.1f} MB')

        print('[build] parsing MRK_List1.rpt...')
        raw = pd.read_csv(
            self.raw_path,
            sep='\t',
            header=0,
            names=self.MRK_LIST1_COLUMNS,
            dtype=str,
            na_values=[''],
            keep_default_na=False,
        )
        print(f'[build] parsed {len(raw):,} rows')

        # Build master table: one row per locus identity. For 'O' (official) status,
        # mgi_id IS the canonical identifier. For 'W' (withdrawn), mgi_id_current
        # points to the replacement.
        master_rows = []
        for _, row in raw.iterrows():
            status = row['status']
            sym = row['symbol']
            syns = row['synonyms']
            syn_list = [] if (pd.isna(syns) or syns == '') else syns.split('|')

            if status == 'O':
                canonical_id = row['mgi_id']
                canonical_symbol = sym
            elif status == 'W':
                # Withdrawn: forward to replacement. If no replacement, skip
                # (locus was withdrawn without a successor)
                canonical_id = row['mgi_id_current']
                canonical_symbol = row['symbol_current']
                if pd.isna(canonical_id) or canonical_id == '':
                    continue
            else:
                # Unknown status code; skip with a warning later
                continue

            master_rows.append({
                'mgi_id':         canonical_id,
                'symbol_current': canonical_symbol,
                'symbol_seen':    sym,
                'status_seen':    status,
                'synonyms':       syn_list,
                'marker_type':    row['marker_type'],
            })

        master_df = pd.DataFrame(master_rows)

        # Collapse to one row per canonical mgi_id, accumulating all observed
        # symbols (current + withdrawn variants + synonyms) into one synonym set
        agg = master_df.groupby('mgi_id', sort=False).agg(
            symbol_current=('symbol_current', 'first'),
            marker_type=('marker_type', 'first'),
            symbols_seen=('symbol_seen', lambda x: list(x)),
            synonyms_lists=('synonyms', lambda x: list(x)),
        )

        all_synonyms = []
        for symbols_seen, synonyms_lists in zip(
            agg['symbols_seen'], agg['synonyms_lists']
        ):
            combined = set()
            combined.update(symbols_seen)
            for sl in synonyms_lists:
                combined.update(sl)
            all_synonyms.append(sorted(s for s in combined if s and not pd.isna(s)))

        agg['all_symbols'] = all_synonyms
        agg = agg.drop(columns=['symbols_seen', 'synonyms_lists'])
        agg = agg.reset_index()
        self.master = agg

        # Build symbol lookup: one edge per (UPPERCASE_SYMBOL, mgi_id, kind)
        lookup_rows = []
        for _, row in agg.iterrows():
            mgi_id = row['mgi_id']
            current = row['symbol_current']
            if current and not pd.isna(current):
                lookup_rows.append({
                    'symbol_upper': current.upper(),
                    'mgi_id':       mgi_id,
                    'kind':         'current',
                })
            for s in row['all_symbols']:
                if s == current:
                    continue
                lookup_rows.append({
                    'symbol_upper': s.upper(),
                    'mgi_id':       mgi_id,
                    'kind':         'synonym',
                })

        lookup_df = pd.DataFrame(lookup_rows).drop_duplicates(
            subset=['symbol_upper', 'mgi_id', 'kind']
        ).reset_index(drop=True)
        self.lookup = lookup_df

        # Persist
        self.master.to_parquet(self.master_path, index=False)
        self.lookup.to_parquet(self.lookup_path, index=False)

        meta = {
            'built_at':       time.strftime('%Y-%m-%d %H:%M:%S'),
            'source_url':     url,
            'raw_size_bytes': os.path.getsize(self.raw_path),
            'n_loci':         int(len(self.master)),
            'n_lookups':      int(len(self.lookup)),
        }
        with open(self.meta_path, 'w') as f:
            json.dump(meta, f, indent=2)

        print(f'[build] master: {len(self.master):,} loci')
        print(f'[build] lookup: {len(self.lookup):,} symbol edges')
        print(f'[build] cache written to {self.cache_dir}')

        return self

    def load(self):
        if not self.is_built():
            raise RuntimeError(
                f'translator not built; call build() first '
                f'(cache_dir={self.cache_dir})'
            )
        self.master = pd.read_parquet(self.master_path)
        self.lookup = pd.read_parquet(self.lookup_path)
        with open(self.meta_path, 'r') as f:
            self.meta = json.load(f)
        print(f'[load] {len(self.master):,} loci, {len(self.lookup):,} symbol edges '
              f'(built {self.meta["built_at"]})')
        return self

    # ---------- TRANSLATE ----------

    def _ensure_loaded(self):
        if self.master is None or self.lookup is None:
            self.load()

    def translate(self, symbol, on_ambiguous='flag'):
        """
        Translate one symbol to its current MGI canonical symbol.

        on_ambiguous: 'flag' | 'strict' | 'best'
            'flag'   - return list of all candidates if multiple matches
            'strict' - return None and warn if multiple matches
            'best'   - prefer 'current' over 'synonym' edges; ties -> first by mgi_id
        """
        self._ensure_loaded()
        if not isinstance(symbol, str) or not symbol.strip():
            return None

        key = symbol.strip().upper()
        hits = self.lookup[self.lookup['symbol_upper'] == key]
        if len(hits) == 0:
            return None

        unique_ids = hits['mgi_id'].unique().tolist()

        if len(unique_ids) == 1:
            mgi_id = unique_ids[0]
            current = self.master.loc[
                self.master['mgi_id'] == mgi_id, 'symbol_current'
            ].iloc[0]
            return current

        # Ambiguous
        candidates = []
        for mgi_id in unique_ids:
            current = self.master.loc[
                self.master['mgi_id'] == mgi_id, 'symbol_current'
            ].iloc[0]
            kinds = hits.loc[hits['mgi_id'] == mgi_id, 'kind'].tolist()
            candidates.append({
                'mgi_id':         mgi_id,
                'symbol_current': current,
                'kinds':          kinds,
            })

        if on_ambiguous == 'flag':
            return candidates
        if on_ambiguous == 'strict':
            warnings.warn(
                f'ambiguous translation for {symbol!r}: '
                f'{[c["symbol_current"] for c in candidates]}'
            )
            return None
        if on_ambiguous == 'best':
            current_hits = [c for c in candidates if 'current' in c['kinds']]
            chosen = current_hits[0] if current_hits else candidates[0]
            return chosen['symbol_current']
        raise ValueError(f'unknown on_ambiguous={on_ambiguous!r}')

    def translate_many(self, symbols, on_ambiguous='flag'):
        """
        Translate a list/Index of symbols to a DataFrame of results.

        Returns DataFrame columns:
            input            - input symbol (verbatim)
            output           - current canonical symbol, or None
            mgi_id           - MGI ID of the resolved locus, or None
            route            - 'identity' (already current), 'case_only',
                               'synonym', 'withdrawn_forwarded',
                               'ambiguous', 'unmapped'
            n_candidates     - number of distinct MGI IDs the input matched
            ambiguous        - True if n_candidates > 1
        """
        self._ensure_loaded()

        symbols = list(symbols)
        results = []
        for s in symbols:
            row = {
                'input':        s,
                'output':       None,
                'mgi_id':       None,
                'route':        'unmapped',
                'n_candidates': 0,
                'ambiguous':    False,
            }
            if not isinstance(s, str) or not s.strip():
                results.append(row)
                continue

            key = s.strip().upper()
            hits = self.lookup[self.lookup['symbol_upper'] == key]
            unique_ids = hits['mgi_id'].unique().tolist()
            row['n_candidates'] = len(unique_ids)

            if len(unique_ids) == 0:
                results.append(row)
                continue

            if len(unique_ids) > 1:
                row['ambiguous'] = True
                if on_ambiguous == 'strict':
                    row['route'] = 'ambiguous'
                    results.append(row)
                    continue
                if on_ambiguous == 'best':
                    current_hits = hits[hits['kind'] == 'current']
                    chosen_mgi = (current_hits['mgi_id'].iloc[0]
                                  if len(current_hits) > 0
                                  else hits['mgi_id'].iloc[0])
                else:  # 'flag'
                    row['route'] = 'ambiguous'
                    chosen_mgi = hits['mgi_id'].iloc[0]
            else:
                chosen_mgi = unique_ids[0]

            mrow = self.master.loc[self.master['mgi_id'] == chosen_mgi].iloc[0]
            current_symbol = mrow['symbol_current']
            row['mgi_id'] = chosen_mgi
            row['output'] = current_symbol

            if not row['ambiguous'] or on_ambiguous == 'best':
                if current_symbol == s:
                    row['route'] = 'identity'
                elif current_symbol.upper() == s.upper():
                    row['route'] = 'case_only'
                else:
                    kinds = hits.loc[hits['mgi_id'] == chosen_mgi, 'kind'].tolist()
                    if 'current' in kinds:
                        row['route'] = 'case_only'
                    else:
                        row['route'] = 'synonym'

            results.append(row)

        return pd.DataFrame(results)

    # ---------- ALIGNMENT HELPERS ----------

    def align_two_indexes(self, idx_a, idx_b, name_a='a', name_b='b',
                          on_ambiguous='best'):
        """
        Translate both indexes to canonical symbols and report cross-set membership.

        Returns:
          translated   - DataFrame with input_a, current_a, input_b, current_b,
                         each row = one canonical locus that appears in at least
                         one input
          summary      - dict with counts: n_a, n_b, n_a_translated, n_b_translated,
                         n_intersection, n_union, n_a_only, n_b_only,
                         n_a_unmapped, n_b_unmapped
        """
        self._ensure_loaded()

        df_a = self.translate_many(idx_a, on_ambiguous=on_ambiguous)
        df_b = self.translate_many(idx_b, on_ambiguous=on_ambiguous)

        a_canon = df_a.dropna(subset=['output']).set_index('output')['input']
        b_canon = df_b.dropna(subset=['output']).set_index('output')['input']

        all_canon = sorted(set(a_canon.index) | set(b_canon.index))
        translated = pd.DataFrame({
            'canonical': all_canon,
            f'input_{name_a}': [a_canon.get(c, None) for c in all_canon],
            f'input_{name_b}': [b_canon.get(c, None) for c in all_canon],
        })
        translated[f'in_{name_a}'] = translated[f'input_{name_a}'].notna()
        translated[f'in_{name_b}'] = translated[f'input_{name_b}'].notna()

        summary = {
            f'n_{name_a}':              int(len(df_a)),
            f'n_{name_b}':              int(len(df_b)),
            f'n_{name_a}_translated':   int(df_a['output'].notna().sum()),
            f'n_{name_b}_translated':   int(df_b['output'].notna().sum()),
            f'n_{name_a}_unmapped':     int(df_a['output'].isna().sum()),
            f'n_{name_b}_unmapped':     int(df_b['output'].isna().sum()),
            f'n_{name_a}_ambiguous':    int(df_a['ambiguous'].sum()),
            f'n_{name_b}_ambiguous':    int(df_b['ambiguous'].sum()),
            'n_intersection':           int(translated[
                f'in_{name_a}'].astype(bool).values
                & translated[f'in_{name_b}'].astype(bool).values).sum().item()
                if hasattr(np, 'sum') else 0,
            'n_union':                  int(len(translated)),
        }
        # Recompute intersection cleanly
        summary['n_intersection'] = int(
            (translated[f'in_{name_a}'] & translated[f'in_{name_b}']).sum()
        )
        summary[f'n_{name_a}_only'] = int(
            (translated[f'in_{name_a}'] & ~translated[f'in_{name_b}']).sum()
        )
        summary[f'n_{name_b}_only'] = int(
            (~translated[f'in_{name_a}'] & translated[f'in_{name_b}']).sum()
        )

        return translated, summary

In [7]:
"""
Build the translator cache. Re-run with force=True only if MGI updates and you
want fresh data. Otherwise this is a one-time cost; subsequent kernels just load.
"""
import os

translator_cache = os.path.join('.', 'gene_translator')

trans = GeneTranslator(cache_dir=translator_cache)
trans.build(force=False)

[build] downloading MRK_List1.rpt...
[build] downloaded 86.3 MB
[build] parsing MRK_List1.rpt...
[build] parsed 734,277 rows
[build] master: 673,518 loci
[build] lookup: 963,910 symbol edges
[build] cache written to ./gene_translator


In [9]:
"""
Spot-check the translator on known cases before running on the real data.
Tests the four failure modes I hypothesized earlier:
  1. case-only difference  (APOE -> Apoe)
  2. classic rename        (Sept7 -> Septin7)
  3. legacy alias          (Apopt1 -> Coa8)
  4. modern symbol stays   (Apoe -> Apoe)
"""
test_cases = [
    ('APOE',     'Apoe'),       # uppercase patch-seq -> title-case canonical
    ('Apoe',     'Apoe'),       # already canonical
    ('Sept7',    'Septin7'),    # known rename
    ('SEPT7',    'Septin7'),    # uppercase + rename
    ('Apopt1',   'Coa8'),       # legacy alias
    ('COA8',     'Coa8'),       # modern uppercase
    ('Ppp2r4',   'Ptpa'),       # another alias from your hand-curated map
    ('PTPA',     'Ptpa'),       # modern uppercase
    ('Tmem246',  'Pgap4'),      # alias
    ('Bogusgene123', None),     # negative control: should be unmapped
]

print(f'{"input":<18} {"expected":<14} {"got":<14} {"status"}')
print('-' * 60)
for sym_in, expected in test_cases:
    got = trans.translate(sym_in, on_ambiguous='best')
    status = 'OK' if got == expected else 'MISMATCH'
    got_str = str(got) if not isinstance(got, list) else f'AMBIG({len(got)})'
    print(f'{sym_in:<18} {str(expected):<14} {got_str:<14} {status}')

input              expected       got            status
------------------------------------------------------------
APOE               Apoe           Apoe           OK
Apoe               Apoe           Apoe           OK
Sept7              Septin7        Septin7        OK
SEPT7              Septin7        Septin7        OK
Apopt1             Coa8           Coa8           OK
COA8               Coa8           Coa8           OK
Ppp2r4             Ptpa           Ptpa           OK
PTPA               Ptpa           Ptpa           OK
Tmem246            Pgap4          Pgap4          OK
Bogusgene123       None           None           OK
